In [73]:
# Libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
import optuna
from sklearn.model_selection import cross_val_score

# For showing all the the Data Frame columns
%matplotlib inline
pd.set_option('display.max_columns', None) 

In [74]:
from utils import *

In [ ]:
# importing reduced dataset
df = pd.read_csv('../data/workable_data.csv')

In [ ]:
df.head()

In [ ]:
# Select target and features
X = df.drop('failure', axis=1)
y = df['failure']

In [ ]:
# Split the data in a way that the test data contains 1 of the failures and the train 3 of the failures
split_index = int(len(df) * 0.7)
X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

In [ ]:
# SMOTE with sampling_strategy = 0.05 
smote = SMOTE(sampling_strategy=0.05, random_state=42) 
X_train, y_train = smote.fit_resample(X_train, y_train)

In [ ]:
# Fit the model
rf = RandomForestClassifier(
    max_depth=10,         
    min_samples_leaf=10,   
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# Feature importance

In [ ]:
sort_idx = rf.feature_importances_.argsort()
plt.barh(X.columns[sort_idx], rf.feature_importances_[sort_idx])
plt.show();

## First Drop

In [ ]:
X = X.drop(['second', 'Pressure_switch', 'Oil_level', 'Caudal_impulses', 'minute', 'hour', 'MPG', 'COMP'], axis=1)

In [ ]:
corr = np.abs(X.corr())

# Set up mask for triangle representation
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(16, 16))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(220, 10, as_cmap=True)

# Draw the heatmap
sns.heatmap(corr, mask=mask, vmax=1, square=True, 
            linewidths=.5, cbar_kws={"shrink": .5},
            annot=True, fmt='.2f',   # fmt='.2f' to round to 2 decimals
            cmap='rocket', ax=ax)

plt.title('Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Split the data in a way that the test data contains 1 of the failures and the train 3 of the failures
split_index = int(len(df) * 0.7)
X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

In [ ]:
# SMOTE with sampling_strategy = 0.05 
smote = SMOTE(sampling_strategy=0.05, random_state=42) 
X_train, y_train = smote.fit_resample(X_train, y_train)

In [ ]:
# Fit the model
rf = RandomForestClassifier(
    max_depth=10,         
    min_samples_leaf=10,   
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

In [ ]:
results = evaluate_model(
    rf,
    X_train, X_test,
    y_train, y_test,
    'Droping_columns',
    'chronological split + SMOTE sampling_strategy=0.0.5 + threshold=0.15',
    threshold=0.15
)

In [ ]:
results

### Conclution
I chose to drop the columns: 'second', 'Pressure_switch', 'Oil_level', 'Caudal_impulses', 'minute', 'hour', 'MPG', 'COMP'

# Cross Validation

In [100]:
df_s = pd.read_csv('../data/reduced_data.csv')

In [ ]:
df_s.columns

In [101]:
df_s = df_s.drop(['timestamp', 'LPS', 'Pressure_switch', 'Oil_level', 
                  'Caudal_impulses', 'minute', 'hour', 'MPG', 'COMP'], axis=1).reset_index(drop=True)

In [ ]:
df_s.head()

In [91]:
df_s.isna().sum()

TP2                0
TP3                0
H1                 0
DV_pressure        0
Reservoirs         0
Oil_temperature    0
Motor_current      0
DV_eletric         0
Towers             0
failure            0
month              0
day                0
dtype: int64

In [102]:
# Select target and features
X = df_s.drop('failure', axis=1)
y = df_s['failure'].astype(int)

In [ ]:
y.dtype

In [103]:
# Split the data in a way that the test data contains 1 of the failures and the train 3 of the failures
split_index = int(len(df_s) * 0.7)
X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

In [ ]:
# from sklearn.metrics import make_scorer, recall_score
# # Define parameter distribution for sampling
# param_dist = {
#     'n_estimators': [100, 200, 300, 500],   # number of trees
#     'max_depth': [None, 10, 20, 30],        # tree depth
#     'min_samples_split': [2, 5, 10],        # min samples to split
#     'min_samples_leaf': [1, 2, 4],          # min samples per leaf
#     'max_features': ['sqrt', 'log2', None]  # number of features per split
# }

# # Create the scorer object
# recall_class1 = make_scorer(recall_score, pos_label=True)
# # Randomized search
# random_search = RandomizedSearchCV(
#     estimator=RandomForestClassifier(
#         class_weight='balanced',
#         random_state=42,
#         n_jobs=-1),
#     param_distributions=param_dist,
#     n_iter=20,                 # number of random combinations to try
#     cv=3,                      # 3-fold cross-validation (faster than 5)
#     scoring=recall_class1,
#     n_jobs=-1,                 # use all CPU cores
#     verbose=2,
#     random_state=42
# )
# # Fit randomized search
# random_search.fit(X_train, y_train)

# # Best parameters
# print("Best parameters:", random_search.best_params_)

# # Best model
# best_rf = random_search.best_estimator_

In [ ]:
# # Evaluate on test set
# y_prob = best_rf.predict_proba(X_test)[:, 1]
# y_pred = (y_prob >= 0.15).astype(int)

# print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score
from imblearn.over_sampling import SMOTE

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    }
    
    rf = RandomForestClassifier(
        **params,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    
    cv = StratifiedKFold(n_splits=3)
    recall_scores = []
    
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_cv_train = X_train.iloc[train_idx]
        X_cv_val   = X_train.iloc[val_idx]
        y_cv_train = y_train.iloc[train_idx]
        y_cv_val   = y_train.iloc[val_idx]
        
        # ✅ SMOTE only on train fold, NEVER on validation fold!
        smote = SMOTE(sampling_strategy=0.05, random_state=42)
        X_cv_train_res, y_cv_train_res = smote.fit_resample(X_cv_train, y_cv_train)
        
        rf.fit(X_cv_train_res, y_cv_train_res)
        
        y_prob = rf.predict_proba(X_cv_val)[:, 1]
        y_pred = (y_prob >= 0.15).astype(int)
        
        # Recall for class 1 only
        score = recall_score(y_cv_val, y_pred, pos_label=1)
        recall_scores.append(score)
    
    return np.mean(recall_scores)

In [80]:
# Create study → direction='maximize' because we want highest recall
study = optuna.create_study(direction='maximize')

# n_trials = how many combinations to try (like n_iter in RandomizedSearch)
study.optimize(objective, show_progress_bar=True)

print("Best parameters:", study.best_params)
print("Best recall score:", study.best_value)

[I 2026-03-12 16:16:52,237] A new study created in memory with name: no-name-d4fe12e6-1af2-4535-86e5-bd76e5b98942
c:\Users\alexc\.conda\envs\final_env\Lib\site-packages\optuna\progress_bar.py:51: UserWarning: Progress bar won't be displayed because n_trials and timeout are None.
  warnings.warn("Progress bar won't be displayed because n_trials and timeout are None.")
[I 2026-03-12 16:17:01,640] Trial 0 finished with value: 0.5413024298293688 and parameters: {'n_estimators': 193, 'max_depth': 21, 'min_samples_split': 4, 'min_samples_leaf': 5, 'max_features': None}. Best is trial 0 with value: 0.5413024298293688.
[I 2026-03-12 16:17:07,501] Trial 1 finished with value: 0.6912108569133251 and parameters: {'n_estimators': 340, 'max_depth': 18, 'min_samples_split': 5, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.6912108569133251.
[I 2026-03-12 16:17:15,035] Trial 2 finished with value: 0.693772085975595 and parameters: {'n_estimators': 445, 'max_depth': 19,

KeyboardInterrupt: 

In [81]:
print("Best parameters:", study.best_params)
print("Best recall score:", study.best_value)

Best parameters: {'n_estimators': 189, 'max_depth': 5, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': None}
Best recall score: 0.7270605544510008


Best parameters: {'n_estimators': 189, 'max_depth': 5, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': None}
Best recall score: 0.7270605544510008

In [104]:
smote = SMOTE(sampling_strategy=0.05, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [105]:
# Train final model with best parameters found
best_rf_optuna = RandomForestClassifier(
    n_estimators=189,
    max_depth=5,
    min_samples_split=4,
    min_samples_leaf=3,
    max_features=None, 
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

best_rf_optuna.fit(X_train_res, y_train_res)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",189
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",4
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",3
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_tr

In [106]:
y_prob = best_rf_optuna.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.15).astype(int)

print(classification_report(y_test, y_pred, zero_division=0))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     75546
           1       0.00      0.00      0.00       270

    accuracy                           1.00     75816
   macro avg       0.50      0.50      0.50     75816
weighted avg       0.99      1.00      0.99     75816



In [107]:
# Diagnose the problem
print("X_train_res shape:", X_train_res.shape)
print("y_train_res shape:", y_train_res.shape)
print("y_train_res dtype:", y_train_res.dtype)
print("y_train_res values:\n", pd.Series(y_train_res).value_counts())
print("\ny_test dtype:", y_test.dtype)
print("y_test values:\n", y_test.value_counts())
print("\nstudy best params:", study.best_params)

X_train_res shape: (180826, 11)
y_train_res shape: (180826,)
y_train_res dtype: int64
y_train_res values:
 failure
0    172216
1      8610
Name: count, dtype: int64

y_test dtype: int64
y_test values:
 failure
0    75546
1      270
Name: count, dtype: int64

study best params: {'n_estimators': 189, 'max_depth': 5, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': None}


In [ ]:
# Plot how each trial performed
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.show()

# Plot which parameters matter most
optuna.visualization.matplotlib.plot_param_importances(study)
plt.show()